# 🛡️ Face Anti-Spoofing — 05: 픽셀 변화율 모듈 (Laplacian + FFT)
> AI Security & Application · 단국대학교 소프트웨어학과  
> 학번: 32214391 · 조현수

---
## 체크리스트
- [ ] Cell 1: Drive 마운트 + 경로 고정
- [ ] Cell 2: 라이브러리 로드
- [ ] Cell 3: Laplacian + FFT 수치 측정 함수
- [ ] Cell 4: 전체 서브셋 수치 측정
- [ ] Cell 5: 공격 유형별 수치 분포 시각화
- [ ] Cell 6: 임계값 설정 + 단독 판정 테스트
- [ ] Cell 7: CNN 앙상블 판정 함수
- [ ] Cell 8: 앙상블 vs 단일 경로 성능 비교

In [ ]:
# ── Cell 1: Drive 마운트 + 경로 고정 ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
BASE       = '/content/drive/MyDrive/face-anti-spoofing'
CROP_DIR   = f'{BASE}/data/cropped'
MODEL_DIR  = f'{BASE}/models'
REPORT_DIR = f'{BASE}/reports'

os.makedirs(REPORT_DIR, exist_ok=True)

print(f'✅ BASE = {BASE}')
for cat in ['live', 'print', 'replay', 'mask']:
    n = len(os.listdir(f'{CROP_DIR}/{cat}'))
    print(f'  cropped/{cat}: {n}장')

In [ ]:
# ── Cell 2: 라이브러리 로드 ───────────────────────────────────
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
import json

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
print('✅ 라이브러리 로드 완료')

In [ ]:
# ── Cell 3: Laplacian + FFT 수치 측정 함수 ───────────────────

def measure_pixel(img_path, target_size=224):
    """
    이미지 경로 → Laplacian 분산 + FFT 고주파 에너지 반환
    
    Laplacian: 경계면 선명도 (낮을수록 블러 = Replay 특징)
    FFT:       고주파 에너지 (인쇄물 망점/화면 격자 패턴 감지)
    """
    img = cv2.imread(img_path)
    if img is None:
        return None, None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (target_size, target_size))

    # Laplacian 분산
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()

    # FFT 고주파 에너지
    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    mag = np.abs(fshift)
    h, w = mag.shape
    cy, cx = h // 2, w // 2
    r = min(h, w) // 8  # 중심 저주파 제거
    mask = np.ones((h, w))
    mask[cy-r:cy+r, cx-r:cx+r] = 0
    fft_energy = np.sum(mag * mask) / (h * w)

    return round(float(lap_var), 2), round(float(fft_energy), 2)


# 함수 테스트
test_cat = 'live'
test_file = os.listdir(f'{CROP_DIR}/{test_cat}')[0]
test_path = f'{CROP_DIR}/{test_cat}/{test_file}'
lap, fft = measure_pixel(test_path)
print(f'테스트 ({test_cat}): Laplacian={lap}, FFT={fft}')
print('✅ 측정 함수 정상')

In [ ]:
# ── Cell 4: 전체 서브셋 수치 측정 ────────────────────────────
# 결과를 CSV로 Drive 저장 → 재실행 시 스킵

CSV_PATH = f'{REPORT_DIR}/pixel_features.csv'

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'✅ 기존 CSV 로드: {len(df)}개')
else:
    records = []
    CAT_TO_BINARY = {'live': 0, 'print': 1, 'replay': 1, 'mask': 1}

    for cat in ['live', 'print', 'replay', 'mask']:
        cat_dir = f'{CROP_DIR}/{cat}'
        files = [f for f in os.listdir(cat_dir)
                 if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

        for i, fname in enumerate(files):
            path = os.path.join(cat_dir, fname)
            lap, fft = measure_pixel(path)
            if lap is not None:
                records.append({
                    'path':     path,
                    'category': cat,
                    'binary':   CAT_TO_BINARY[cat],
                    'laplacian': lap,
                    'fft':      fft
                })
            if (i + 1) % 200 == 0:
                print(f'  [{cat}] {i+1}/{len(files)} 완료')

        print(f'✅ [{cat}] 측정 완료')

    df = pd.DataFrame(records)
    df.to_csv(CSV_PATH, index=False)
    print(f'\n✅ CSV 저장: {CSV_PATH}')

print(f'\n📊 데이터 요약')
print(df.groupby('category')[['laplacian', 'fft']].mean().round(1))

In [ ]:
# ── Cell 5: 공격 유형별 수치 분포 시각화 ─────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Pixel Features by Attack Type', fontsize=14)

COLORS = {'live': '#2ecc71', 'print': '#e74c3c',
          'replay': '#3498db', 'mask': '#9b59b6'}

# Laplacian 분포
for cat in ['live', 'print', 'replay', 'mask']:
    data = df[df['category'] == cat]['laplacian']
    axes[0].hist(data, bins=40, alpha=0.6,
                 label=cat, color=COLORS[cat])
axes[0].set_title('Laplacian Variance Distribution')
axes[0].set_xlabel('Laplacian Variance (higher = sharper)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# FFT 분포
for cat in ['live', 'print', 'replay', 'mask']:
    data = df[df['category'] == cat]['fft']
    axes[1].hist(data, bins=40, alpha=0.6,
                 label=cat, color=COLORS[cat])
axes[1].set_title('FFT High-Frequency Energy Distribution')
axes[1].set_xlabel('FFT Energy')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/pixel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# 카테고리별 평균 + 표준편차
print('\n📊 카테고리별 수치 통계')
stats = df.groupby('category')[['laplacian', 'fft']].agg(['mean', 'std']).round(1)
print(stats)

In [ ]:
# ── Cell 6: 임계값 설정 + 단독 판정 테스트 ───────────────────
# 02_frequency_analysis 결과 + Cell 5 통계 기반 임계값 설정

from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

# 임계값 (Live vs Spoof 이진 판정)
# Replay는 Laplacian이 낮음 → 낮으면 Spoof 의심
# Print는 FFT가 높음 → 높으면 Spoof 의심
LAP_THRESHOLD = df[df['category'] == 'live']['laplacian'].quantile(0.25)
FFT_THRESHOLD = df[df['category'] == 'live']['fft'].quantile(0.75)

print(f'Laplacian 임계값 (이하 = Spoof 의심): {LAP_THRESHOLD:.1f}')
print(f'FFT 임계값 (이상 = Spoof 의심):       {FFT_THRESHOLD:.1f}')

def pixel_predict(lap, fft):
    """픽셀 수치만으로 Live(0) / Spoof(1) 판정"""
    if lap < LAP_THRESHOLD or fft > FFT_THRESHOLD:
        return 1  # Spoof
    return 0  # Live

# 전체 테스트
df['pixel_pred'] = df.apply(
    lambda r: pixel_predict(r['laplacian'], r['fft']), axis=1
)

pixel_acc = accuracy_score(df['binary'], df['pixel_pred'])
print(f'\n픽셀 모듈 단독 Accuracy: {pixel_acc:.4f}')

# Confusion Matrix
cm = confusion_matrix(df['binary'], df['pixel_pred'])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', ax=ax,
            xticklabels=['Live', 'Spoof'],
            yticklabels=['Live', 'Spoof'],
            cmap='Blues')
ax.set_title('Pixel Module Confusion Matrix')
ax.set_ylabel('True')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/pixel_confusion.png', dpi=150)
plt.show()

# FAR / FRR
live_df  = df[df['binary'] == 0]
spoof_df = df[df['binary'] == 1]
FAR = (spoof_df['pixel_pred'] == 0).sum() / len(spoof_df)
FRR = (live_df['pixel_pred'] == 1).sum() / len(live_df)
print(f'FAR (가짜가 통과): {FAR:.4f} ({FAR*100:.1f}%)')
print(f'FRR (진짜가 거부): {FRR:.4f} ({FRR*100:.1f}%)')

In [ ]:
# ── Cell 7: CNN + 픽셀 앙상블 판정 함수 ──────────────────────

# CNN 모델 로드
model = tf.keras.models.load_model(f'{MODEL_DIR}/stage2_best.h5')
print('✅ CNN 모델 로드 완료')

IMG_MEAN = np.array([0.485, 0.456, 0.406])
IMG_STD  = np.array([0.229, 0.224, 0.225])

SPOOF_LABELS = {0: 'Live', 1: 'Print', 2: 'Replay', 3: 'Mask'}

def predict_single(img_path, cnn_weight=0.7, pixel_weight=0.3):
    """
    단일 이미지 → 앙상블 판정
    반환: {
        'final': 'REAL' or 'FAKE',
        'cnn_prob': float,
        'spoof_type': str,
        'laplacian': float,
        'fft': float,
        'pixel_pred': 0 or 1,
        'ensemble_score': float
    }
    """
    # CNN 경로
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (224, 224))
    img_norm = (img_resized / 255.0 - IMG_MEAN) / IMG_STD
    img_tensor = np.expand_dims(img_norm, axis=0).astype(np.float32)

    pred_binary, pred_spoof = model.predict(img_tensor, verbose=0)
    cnn_prob   = float(pred_binary[0][0])   # Spoof 확률
    spoof_type = SPOOF_LABELS[np.argmax(pred_spoof[0])]

    # 픽셀 경로
    lap, fft = measure_pixel(img_path)
    pixel_pred = pixel_predict(lap, fft)

    # 앙상블
    ensemble_score = cnn_prob * cnn_weight + pixel_pred * pixel_weight
    final = 'FAKE' if ensemble_score > 0.5 else 'REAL'

    return {
        'final':          final,
        'cnn_prob':       round(cnn_prob, 4),
        'spoof_type':     spoof_type,
        'laplacian':      lap,
        'fft':            fft,
        'pixel_pred':     pixel_pred,
        'ensemble_score': round(ensemble_score, 4)
    }

# 테스트
for cat in ['live', 'print', 'replay', 'mask']:
    sample = os.listdir(f'{CROP_DIR}/{cat}')[0]
    result = predict_single(f'{CROP_DIR}/{cat}/{sample}')
    print(f'[{cat}] → {result["final"]} '
          f'(CNN:{result["cnn_prob"]:.2f} '
          f'Lap:{result["laplacian"]} '
          f'FFT:{result["fft"]} '
          f'Type:{result["spoof_type"]})')

In [ ]:
# ── Cell 8: 앙상블 vs 단일 경로 성능 비교 ────────────────────

from sklearn.metrics import accuracy_score, roc_auc_score

# 테스트셋 구성 (카테고리당 100장)
TEST_N = 100
test_records = []

CAT_TO_BINARY = {'live': 0, 'print': 1, 'replay': 1, 'mask': 1}

for cat in ['live', 'print', 'replay', 'mask']:
    files = os.listdir(f'{CROP_DIR}/{cat}')[:TEST_N]
    for fname in files:
        path = f'{CROP_DIR}/{cat}/{fname}'
        result = predict_single(path)
        test_records.append({
            'category':       cat,
            'true_binary':    CAT_TO_BINARY[cat],
            'cnn_pred':       1 if result['cnn_prob'] > 0.5 else 0,
            'cnn_prob':       result['cnn_prob'],
            'pixel_pred':     result['pixel_pred'],
            'ensemble_pred':  1 if result['ensemble_score'] > 0.5 else 0,
            'ensemble_score': result['ensemble_score']
        })
    print(f'✅ [{cat}] {len(files)}장 완료')

test_df = pd.DataFrame(test_records)

# 성능 비교
cnn_acc      = accuracy_score(test_df['true_binary'], test_df['cnn_pred'])
pixel_acc    = accuracy_score(test_df['true_binary'], test_df['pixel_pred'])
ensemble_acc = accuracy_score(test_df['true_binary'], test_df['ensemble_pred'])

cnn_auc      = roc_auc_score(test_df['true_binary'], test_df['cnn_prob'])
ensemble_auc = roc_auc_score(test_df['true_binary'], test_df['ensemble_score'])

print('\n' + '=' * 45)
print('📊 성능 비교')
print('=' * 45)
print(f'{'방법':<15} {'Accuracy':>10} {'ROC-AUC':>10}')
print('-' * 45)
print(f'{'CNN 단독':<15} {cnn_acc:>10.4f} {cnn_auc:>10.4f}')
print(f'{'픽셀 단독':<15} {pixel_acc:>10.4f} {"N/A":>10}')
print(f'{'앙상블':<15} {ensemble_acc:>10.4f} {ensemble_auc:>10.4f}')

# 공격 유형별 FAR
print('\n📊 공격 유형별 FAR (앙상블)')
for cat in ['print', 'replay', 'mask']:
    cat_df = test_df[test_df['category'] == cat]
    far = (cat_df['ensemble_pred'] == 0).sum() / len(cat_df)
    print(f'  {cat:<8}: FAR = {far:.4f} ({far*100:.1f}%)')

# 결과 저장
comparison = {
    'cnn_accuracy':      round(cnn_acc, 4),
    'pixel_accuracy':    round(pixel_acc, 4),
    'ensemble_accuracy': round(ensemble_acc, 4),
    'cnn_auc':           round(cnn_auc, 4),
    'ensemble_auc':      round(ensemble_auc, 4)
}
with open(f'{REPORT_DIR}/ensemble_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)

print(f'\n✅ 결과 저장: {REPORT_DIR}/ensemble_comparison.json')

## ✅ 완료 체크리스트

| 항목 | 상태 |
|------|------|
| Drive 마운트 + 경로 고정 | ⬜ |
| Laplacian + FFT 함수 구현 | ⬜ |
| 전체 서브셋 수치 측정 + CSV 저장 | ⬜ |
| 공격 유형별 분포 시각화 | ⬜ |
| 임계값 설정 + 단독 판정 테스트 | ⬜ |
| CNN + 픽셀 앙상블 함수 구현 | ⬜ |
| 앙상블 vs 단일 경로 성능 비교 | ⬜ |

### 다음 단계
- **06_streamlit_app.py:** Streamlit MVP 앱 구현
  - 이미지 업로드 UI
  - CNN + 픽셀 앙상블 판정 결과 표시
  - Spoof Type 예측 표시
  - Laplacian / FFT 수치 표시